In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime

## VV

In [2]:
ds_vv = xr.open_dataset('/data/shared_data/ARM_data/SGP/others/sgpdlprofwstats4newsC1.c1/sgpdlprofwstats4newsC1.c1.20220402.000000.custom.nc')

### Extracting 750–850 hPa Mean Vertical Velocity from ENA Doppler Lidar Data, if you want to get the result for SGP, replace any 'ENA' to 'SGP'

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import datetime
import re

# ===================== User paths =====================
base_dir = Path("/data/shared_data/ARM_data/ENA/others/enadlprofwstats4newsC1.c1")
out_dir  = Path("/data/ggong/ARM_monthly/ENA/VV_750_850")
out_dir.mkdir(parents=True, exist_ok=True)

# ===================== Helper functions =====================
def convert_time_to_datetime(ds, date_str):
    times = pd.date_range(start=pd.Timestamp(date_str), periods=144, freq="10min")
    ds = ds.assign_coords(time=("time", times))

    # Remove existing units to avoid encoding conflicts
    ds.time.attrs.pop("units", None)
    ds.time.encoding.pop("units", None)
    return ds

def convert_height_to_coords(ds, start=0.015, step=0.03, unit="km"):
    n = ds.sizes["height"]  # Use the actual number of height levels
    new_height = start + step * np.arange(n, dtype=float)
    ds = ds.assign_coords(height=("height", new_height))
    ds.height.attrs = {"units": unit, "long_name": "Geometric height"}
    return ds

def compute_w_mean(ds, hmin=1.4567, hmax=2.4652):
    w_mean = ds["w"].sel(height=slice(hmin, hmax)).mean(dim="height", skipna=True)
    w_mean.name = "w_750_850"
    w_mean.attrs = {"units": "m/s", "long_name": f"Mean w between {hmin}-{hmax} km"}
    return w_mean

# ===================== Batch processing =====================
files = sorted(base_dir.glob("*.nc"))
count = 0

for i, f in enumerate(files, 1):
    try:
        ds = xr.open_dataset(f)

        # Convert height coordinate
        ds = convert_height_to_coords(ds)

        # Extract date from filename and convert time coordinate
        date_match = re.search(r"\.(\d{8})\.", str(f))
        if not date_match:
            raise ValueError(f"Could not extract date from filename: {f}")

        date_str = date_match.group(1)
        ds = convert_time_to_datetime(ds, date_str)

        # Compute height-mean vertical velocity
        if "w" not in ds:
            raise KeyError(f"Variable 'w' not found in file: {f}")

        w_mean = compute_w_mean(ds)

        # Save output file
        out_path = out_dir / f"VV_mean_750_850_{date_str}.nc"
        w_mean.to_netcdf(out_path)
        count += 1

        # Print progress every 300 files
        if i % 300 == 0:
            print(f"[{datetime.now()}] Completed {i} files")

    except Exception as e:
        print(f"[ERROR] {f}: {e}")

    finally:
        try:
            ds.close()
        except Exception:
            pass

print(f"Done. Successfully generated {count} output files.")

In [2]:
ds_vvv=xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/VV_750_850/*')

In [ ]:
import pandas as pd

START = "2016-01-01 00:00:00"
END   = "2025-12-31 23:59:00"

# Fill missing 10-min timestamps with NaN while preserving the original 10-min grid
full_10min = pd.date_range(START, END, freq="10min")
ds_10 = ds_vvv.sortby("time").reindex(time=full_10min)

# Create the target 2-min time grid
target_2min = pd.date_range(START, END, freq="2min")

# Linearly interpolate from 10-min to 2-min resolution.
# No extrapolation is applied outside the data range, so out-of-range values remain NaN.
ds_2min = ds_10.interp(time=target_2min, method="linear")

In [ ]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/VV_2min_750_850_extend.nc')

### Extracting 600 hPa (4 km) Mean Vertical Velocity from SGP Doppler Lidar Data, if you want to get the result for ENA, replace any 'SGP' to 'ENA'¶

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import datetime
import re

# ===================== Paths =====================
base_dir = Path("/data/shared_data/ARM_data/SGP/others/sgpdlprofwstats4newsC1.c1")
out_dir  = Path("/data/ggong/ARM_monthly/SGP/VV_600")
out_dir.mkdir(parents=True, exist_ok=True)

# ===================== Functions =====================
def convert_time_to_datetime(ds, date_str):
    """Set time as a 10-min resolution time coordinate for the day with 144 points, and remove old units to avoid conflicts."""
    times = pd.date_range(start=pd.Timestamp(date_str), periods=144, freq="10min")
    ds = ds.assign_coords(time=("time", times))
    # Clean potential units conflicts
    if "units" in ds.time.attrs:
        ds.time.attrs.pop("units", None)
    if "units" in ds.time.encoding:
        ds.time.encoding.pop("units", None)
    return ds

def convert_height_to_coords(ds, start=0.015, step=0.03, unit="km"):
    """
    Convert height to geometric height coordinates in km.
    Use the original number of levels n, starting from start with interval step.
    """
    n = ds.sizes["height"]
    new_height = (start + step * np.arange(n, dtype=float))
    ds = ds.assign_coords(height=("height", new_height))
    ds.height.attrs = {"units": unit, "long_name": "Geometric height"}
    return ds

def compute_w_600(ds):
    """
    If the number of height levels is 133, extract w from the highest level, isel(height=-1).
    Otherwise, return all NaN with the same time dimension.
    The output only has the time dimension.
    """
    if "w" not in ds:
        raise KeyError("Variable 'w' is not found in the dataset")
    if "height" not in ds.dims or "time" not in ds.dims:
        raise KeyError("Dataset is missing required dimensions: 'height' or 'time'")

    if ds.sizes.get("height", -1) == 133:
        # Directly select the highest level; drop=True removes the height dimension and keeps only time
        da = ds["w"].isel(height=-1, drop=True)
    else:
        # Generate a NaN series aligned with time
        da = xr.DataArray(
            np.full(ds.sizes["time"], np.nan, dtype=float),
            coords={"time": ds["time"]},
            dims=("time",),
        )

    da.name = "w_600"
    da.attrs = {
        "units": "m/s",
        "long_name": "Top-level w when height has 133 levels; otherwise NaN",
        "note": "Top level corresponds to isel(height=-1) after converting geometric height coords.",
    }
    return da

# ===================== Batch processing =====================
files = sorted(base_dir.glob("*.nc"))
count = 0

for i, f in enumerate(files, 1):
    ds = None
    try:
        ds = xr.open_dataset(f)

        # Extract date from filename, formatted as .YYYYMMDD.
        date_match = re.search(r"\.(\d{8})\.", str(f))
        if not date_match:
            raise ValueError(f"Could not extract date: {f}")
        date_str = date_match.group(1)

        # Convert height and time coordinates
        ds = convert_height_to_coords(ds)          # Still uses start=0.015 km and step=0.03 km
        ds = convert_time_to_datetime(ds, date_str)

        # Compute w_600
        w_600 = compute_w_600(ds)

        # Save output
        out_path = out_dir / f"VV_600_{date_str}.nc"
        w_600.to_netcdf(out_path)
        count += 1

        # Print progress
        if i % 300 == 0:
            print(f"[{datetime.now()}] Completed {i} files")

    except Exception as e:
        print(f"[ERROR] {f}: {e}")

    finally:
        try:
            if ds is not None:
                ds.close()
        except Exception:
            pass

print(f"Done. A total of {count} files were generated.")

In [39]:
ds_vvv=xr.open_mfdataset('/data/ggong/ARM_monthly/SGP/VV_600/*')

In [ ]:
import pandas as pd

START = "2016-01-01 00:00:00"
END   = "2025-12-31 23:58:00"

# Fill missing 10-min timestamps with NaN while preserving the original 10-min grid
full_10min = pd.date_range(START, END, freq="10min")
ds_10 = ds_vvv.sortby("time").reindex(time=full_10min)

# Target 2-min time grid
target_2min = pd.date_range(START, END, freq="2min")

# Linearly interpolate to 2-min resolution along time
# No extrapolation outside the data range -> out-of-range values remain NaN
ds_2min = ds_10.interp(time=target_2min, method="linear")

In [42]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/SGP/VV_2min_600_extend.nc')